[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/06_strukturierte_ausgabe.ipynb)

# Sitzung 6 — Strukturierte Ausgabe (und Embeddings zum Anfassen)

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Ein LLM, das in Prosa antwortet, ist für ein Programm wertlos. Wir brauchen **verlässliche, maschinenlesbare Ausgabe** — jedes Mal. Heute: warum das kniffliger ist, als es klingt, und wie man sich absichert. Danach: Embeddings, diesmal *echt* gerechnet.

## 0. Setup

In [ ]:
import json, re, random
print('Fertig.')

## 1. Warum überhaupt JSON?

Bisher haben wir Ergebnisse *angeschaut*. Aber ein Dashboard, eine Datenbank, der nächste Schritt in der Pipeline — die brauchen **Struktur**, kein Fließtext. JSON ist das Standard-Format dafür:

```json
{"sentiment": "negative", "themes": ["Akku", "Preis"]}
```

Das Problem: ein LLM ist ein *Text*-Modell. Es gibt keine Garantie, dass es sauberes JSON zurückgibt — mal packt es Erklärungen drumherum, mal Code-Zäune, mal ein fehlendes Komma. **Und ein einziger Formatfehler bricht die ganze Verarbeitung.**

## 2. Wenn die Ausgabe bricht

Damit ihr das *seht*, haben wir einen Mock gebaut, der sich wie ein echtes LLM **unzuverlässig** verhält: mal sauberes JSON, mal mit Prosa drumherum, mal in Code-Zäunen, mal kaputt. Führt die Zelle mehrmals aus:

In [ ]:
import random

def mock_llm_rohausgabe(text):
    '''Simuliert ein LLM, das NICHT immer sauberes JSON liefert.'''
    kern = '{"sentiment": "negative", "themes": ["Akku"]}'
    variante = random.choice([
        kern,                                              # sauber
        f'Gerne! Hier das Ergebnis:\n{kern}',              # Prosa davor
        f'```json\n{kern}\n```',                          # Code-Zaun
        '{"sentiment": "negative", "themes": ["Akku",]}', # Trailing Comma (kaputt)
        f'{kern}\nHoffe, das hilft!',                      # Prosa danach
    ])
    return variante

for _ in range(5):
    print(repr(mock_llm_rohausgabe('...')))
    print('---')

## 3. Der naive Weg bricht

Der erste Reflex: `json.loads(...)` direkt auf die Ausgabe. Führt das mehrmals aus — irgendwann kracht es:

In [ ]:
roh = mock_llm_rohausgabe('...')
print('Rohausgabe:', repr(roh))
daten = json.loads(roh)   # bricht bei Prosa / Code-Zaun / Trailing Comma
print('Geparst:', daten)

> ⚠️ Je nach Variante: `JSONDecodeError`. In einer Schleife über 300 Bewertungen würde **ein** solcher Fehler alles stoppen. Das ist inakzeptabel für ein Produkt.

## 4. Verlässlich parsen: extrahieren, validieren, absichern

Die Lösung ist **defensives Parsen**: nicht blind vertrauen, sondern das JSON aus dem Text herausschneiden, versuchen zu laden, und bei Fehler einen sauberen Fallback liefern — statt abzustürzen.

In [ ]:
def safe_json(roh):
    '''Schneidet das JSON heraus, parst defensiv, faellt sauber zurueck.'''
    # 1. das erste {...} im Text finden (ignoriert Prosa + Code-Zaeune)
    m = re.search(r'\{.*\}', roh, re.DOTALL)
    if not m:
        return {'sentiment': 'unknown', 'themes': [], '_fehler': 'kein JSON'}
    text = m.group(0)
    # 2. haeufige Fehler reparieren (Trailing Comma)
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # 3. versuchen zu laden
    try:
        daten = json.loads(text)
    except json.JSONDecodeError:
        return {'sentiment': 'unknown', 'themes': [], '_fehler': 'kaputt'}
    # 4. validieren: Pflichtfelder sicherstellen
    daten.setdefault('sentiment', 'unknown')
    daten.setdefault('themes', [])
    return daten

# Jetzt bricht nichts mehr — egal welche Variante:
for _ in range(8):
    roh = mock_llm_rohausgabe('...')
    print(safe_json(roh))

> 💡 **Kernpunkt:** Nie darauf vertrauen, dass ein LLM sauberes Format liefert. **Extrahieren, reparieren, validieren, absichern.** Ein Produkt darf nicht an einem fehlenden Komma sterben.

## 5. Eure Aufgabe

Erweitert `mock_llm_rohausgabe` um eine **neue kaputte Variante** (z. B. einfache statt doppelte Anführungszeichen, oder ganz ohne JSON) und prüft: fängt `safe_json` sie sauber ab? Wenn nicht — wie müsste man den Parser härten?

In [ ]:
# Beispiel: eine fiese neue Variante zum Testen
fies = "Das Modell sagt: leider negativ, keine klaren Themen."  # gar kein JSON
print(safe_json(fies))

## Der tiefere Punkt: Embeddings zum Anfassen

Letzte Woche die Idee: **Bedeutung als Geometrie** — ähnliche Wörter liegen nah beieinander. Heute rechnen wir das *echt* aus.

Wir laden ein kleines Sprachmodell, das jedes Wort in einen **Vektor** (eine Liste von Zahlen) übersetzt. Dann messen wir die **Ähnlichkeit** zwischen Wörtern — und ihr seht: „Akku“ und „Batterie“ rücken zusammen, „Preis“ liegt weit weg.

> ⏱️ **Hinweis:** Die nächste Zelle lädt einmalig ein kleines Modell (~1 Minute). Das ist ein **Demo** — zum Anschauen und Diskutieren, nicht zum Selbstbauen.

In [ ]:
# Einmalig: kleines Embedding-Modell laden (Download ~90 MB)
!pip install sentence-transformers --quiet
from sentence_transformers import SentenceTransformer
modell = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('Modell geladen.')

In [ ]:
import numpy as np

woerter = ['Akku', 'Batterie', 'Klang', 'Ton', 'Sound',
           'Preis', 'Geld', 'Komfort', 'bequem', 'App']
vektoren = modell.encode(woerter, normalize_embeddings=True)

# Aehnlichkeit = Skalarprodukt (bei normalisierten Vektoren = Kosinus-Aehnlichkeit)
aehnlich = vektoren @ vektoren.T

def sim(a, b):
    return round(float(aehnlich[woerter.index(a)][woerter.index(b)]), 2)

print('Akku  ~ Batterie:', sim('Akku','Batterie'), '  (nah = aehnlich)')
print('Klang ~ Ton     :', sim('Klang','Ton'))
print('Klang ~ Sound   :', sim('Klang','Sound'))
print('Preis ~ Geld    :', sim('Preis','Geld'))
print('Akku  ~ Preis   :', sim('Akku','Preis'), '  (weit weg = unaehnlich)')
print('Akku  ~ Komfort :', sim('Akku','Komfort'))

### Die Landkarte der Bedeutungen

Vektoren haben viele Dimensionen — zu viele zum Zeichnen. Mit einem Trick (PCA) projizieren wir sie auf 2 Dimensionen und *sehen* die Nähe:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

xy = PCA(n_components=2).fit_transform(vektoren)
plt.figure(figsize=(8,6))
plt.scatter(xy[:,0], xy[:,1], s=40)
for i, w in enumerate(woerter):
    plt.annotate(w, (xy[i,0], xy[i,1]), fontsize=13,
                 xytext=(6,3), textcoords='offset points')
plt.title('Bedeutung als Geometrie — nahe Woerter sind aehnlich')
plt.axis('off'); plt.show()

> 💡 **Kernpunkt:** Genau *das* nutzt ein LLM, um „Ton“ und „Klang“ als dasselbe Thema zu erkennen — ohne Stichwort-Liste. Bedeutung ist Nähe im Raum. **Diskussion:** Wo würde man Embeddings statt eines LLM einsetzen — und umgekehrt? (Embeddings: schnell, billig, aber gröber. LLM: versteht Kontext, aber teurer/langsamer.)

## 6. Geschafft — Meilenstein-Vorbereitung

Ihr habt jetzt: verlässliche strukturierte Ausgabe (die Basis jeder Pipeline) und ein echtes Gefühl für Embeddings.

**Nächste Woche (18.11):** Aggregation & Trends — Sentiment über die Zeit, Top-Themen, das Gesamtbild verdichten. Und in Sitzung 8: alles zum **fertigen Dashboard** zusammensetzen.